# Fixed-subset eval: compare a submission before spending a Kaggle submit

Runs one `submission.zip` on the **same** tasks from `tasks.jsonl` every time, so two variants can be compared
on identical tasks, and v0's score here can be compared with its leaderboard score.

**Inputs to attach:** the competition data, the `metric/gemma-4-developer-agent-wheelhouse` dataset, the Gemma 4
model matching `agent.yaml`, and a private dataset holding the `submission.zip` to test.
**Accelerator:** L4x4 (counts double against GPU quota). **Internet:** off.

First run: leave `FIXED_SUBSET_IDS = None`; the notebook picks a subset and prints its ids.
Paste those ids into `FIXED_SUBSET_IDS` for every later run, and commit them to the repo.

## 1. Install the wheelhouse (from the starter notebook)

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Unpack the submission and load tasks

In [ ]:
import collections
import json
import re
import zipfile
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')

RUN_NAME = 'v1'
DS_CANDIDATES = [
    Path('/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/gemma4-e4b-sub'),
    Path('/kaggle/input/gemma4-e4b-sub'),
]
DS_ROOT = next((p for p in DS_CANDIDATES if p.exists()), None)
assert DS_ROOT is not None, f'Submission dataset not found in {DS_CANDIDATES}'

SUBSET_SIZE = 12
SUBSET_SEED = 20261120
FIXED_SUBSET_IDS = ["fastapi_13537", "fastapi_14349", "fastapi_14873", "fastapi_14964", "fastapi_14978", "httpx_3672", "requests_6644", "requests_7502", "rich_3063", "rich_3468", "rich_3518", "rich_3782"]  # stratified 12-task val split (seed 20261120)

RUN_DIR = WORKING_DIR / RUN_NAME
AGENT_DIR = RUN_DIR / 'agent'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True)

if (DS_ROOT / 'submission.zip').is_file():
    with zipfile.ZipFile(DS_ROOT / 'submission.zip') as zf:
        zf.extractall(AGENT_DIR)
else:
    shutil.copytree(DS_ROOT, AGENT_DIR, dirs_exist_ok=True)
assert (AGENT_DIR / 'agent.yaml').is_file(), f'agent.yaml missing in {AGENT_DIR}: {list(AGENT_DIR.iterdir())}'

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
by_repo = collections.Counter(t.repo for t in tasks)
print(f'{len(tasks)} tasks in {TASKS_PATH.name}, {len(by_repo)} repos')
for repo, n in by_repo.most_common():
    print(f'  {n:4d}  {repo}')
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')


## 3. Fixed task subset

In [ ]:
import random


def pick_subset(tasks, n, seed):
    """Stratified subset by repo (proportional, >=1 each) and patch-size quantiles."""
    rng = random.Random(seed)
    by_repo = collections.defaultdict(list)
    for t in tasks:
        by_repo[t.repo].append(t)
    for repo in by_repo:
        by_repo[repo].sort(key=lambda t: (len(getattr(t, 'patch', '') or ''), t.instance_id))

    repos = sorted(by_repo.keys(), key=lambda r: (-len(by_repo[r]), r))
    counts = {}
    if n >= len(repos):
        for r in repos:
            counts[r] = 1
        rem = n - len(repos)
        weights = {r: len(by_repo[r]) - 1 for r in repos}
        w_sum = sum(weights.values()) or 1
        fracs = []
        for r in repos:
            exact = rem * weights[r] / w_sum
            add = min(int(exact), len(by_repo[r]) - counts[r])
            counts[r] += add
            fracs.append((exact - add, r))
        left = n - sum(counts.values())
        for _, r in sorted(fracs, reverse=True):
            if left <= 0:
                break
            if counts[r] < len(by_repo[r]):
                counts[r] += 1
                left -= 1
    else:
        for r in repos[:n]:
            counts[r] = 1

    chosen = []
    for r in repos:
        k = counts.get(r, 0)
        pool = by_repo[r]
        if k <= 0:
            continue
        if k >= len(pool):
            chosen.extend(pool)
            continue
        for i in range(k):
            lo = int(i * len(pool) / k)
            hi = max(lo + 1, int((i + 1) * len(pool) / k))
            chosen.append(rng.choice(pool[lo:hi]))
    return chosen


if FIXED_SUBSET_IDS:
    wanted = set(FIXED_SUBSET_IDS)
    subset = [t for t in tasks if t.instance_id in wanted]
    missing = wanted - {t.instance_id for t in subset}
    assert not missing, f'ids not in tasks.jsonl: {sorted(missing)}'
else:
    subset = pick_subset(tasks, SUBSET_SIZE, SUBSET_SEED)
subset = sorted(subset, key=lambda t: t.instance_id)
ids = [t.instance_id for t in subset]
(RUN_DIR / 'subset_ids.json').write_text(json.dumps(ids, indent=1))
print(f'{len(subset)} tasks. Paste into FIXED_SUBSET_IDS for later runs:')
print(json.dumps(ids))


## 4. Start vLLM (from the starter notebook)

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 5. Run the subset

In [ ]:
import asyncio
import concurrent.futures
import time
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


eval_config_file = next((c for name in ('eval_config.yaml', 'eval_config.yml')
                         if (c := AGENT_DIR / name).is_file()), None)
raw = (yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}) if eval_config_file else {}
ev = raw.get('evaluation', raw)
max_time_minutes = float(ev.get('max_time_minutes', 60.0))
turns_raw = ev.get('max_turns', ev.get('max_llm_calls'))
limits, gen_constraints = build_submission_limits()

eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RUN_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=int(ev.get('timeout_seconds', 300)),
    verification_timeout_seconds=int(ev.get('verification_timeout_seconds', 300)),
    max_time_minutes=max_time_minutes,
    max_tool_calls=int(ev.get('max_tool_calls', 100)),
    max_turns=int(turns_raw) if turns_raw is not None else None,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)
evaluator = Evaluator(eval_config)

rows_path = RUN_DIR / 'per_task.jsonl'
rows_path.unlink(missing_ok=True)
rows = []
for idx, task in enumerate(subset, start=1):
    started = time.perf_counter()
    try:
        result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(subset))
        patch = (result.agent_patch or '').strip()
        row = dict(id=task.instance_id, repo=task.repo, resolved=bool(result.resolved),
                   patch_chars=len(patch), tool_calls=result.tool_calls,
                   agent_seconds=round(float(result.duration_seconds or 0.0), 1),
                   test_exit_code=result.test_exit_code, error=getattr(result, 'error', None))
    except Exception as exc:  # one broken task must not lose the others
        row = dict(id=task.instance_id, repo=task.repo, resolved=False, patch_chars=0,
                   tool_calls=None, agent_seconds=None, test_exit_code=None, error=repr(exc)[:300])
    row['wall_seconds'] = round(time.perf_counter() - started, 1)
    rows.append(row)
    with rows_path.open('a') as f:
        f.write(json.dumps(row) + '\n')
    print(f"[{idx}/{len(subset)}] {row['id']}: resolved={row['resolved']} patch={row['patch_chars']} "
          f"agent={row['agent_seconds']}s wall={row['wall_seconds']}s err={row['error']}")

## 6. Summary

In [ ]:
df = pd.DataFrame(rows)
agent_s = df['agent_seconds'].dropna()
cap = max_time_minutes * 60
summary = {
    'run': RUN_NAME,
    'tasks': len(df),
    'resolved': int(df['resolved'].sum()),
    'resolution_rate': round(float(df['resolved'].mean()), 3),
    'no_patch': int((df['patch_chars'] == 0).sum()),
    'errors': int(df['error'].notna().sum()),
    'hit_time_cap': int((agent_s >= 0.95 * cap).sum()),
    'agent_seconds_median': float(agent_s.median()) if len(agent_s) else None,
    'agent_seconds_max': float(agent_s.max()) if len(agent_s) else None,
    'wall_seconds_mean': round(float(df['wall_seconds'].mean()), 1),
    # Only meaningful if scoring runs tasks one after another, which HARNESS_README does not say.
    'tasks_fitting_12h_if_sequential': int(12 * 3600 // df['wall_seconds'].mean()) if len(df) else None,
}
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
print(df.to_string(index=False))
display(df)
